# SVM Model Training and Evaluation

This notebook trains a Support Vector Machine (SVM) on the preprocessed CICEVSE2024 dataset. It includes data loading, evaluation, and hyperparameter tuning using `GridSearchCV`.

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Ensure plots are displayed inline
%matplotlib inline

## 1. Load Data
Load the preprocessed train, validation, and test datasets. Note: ensure `preprocess.py` has been run previously.

In [ ]:
# Adjust path assuming the notebook runs from the project root or src/models/svm
import sys
if os.path.exists('../../../data/processed'):
    DATA_DIR = '../../../data/processed'
elif os.path.exists('data/processed'):
    DATA_DIR = 'data/processed'
else:
    DATA_DIR = '../data/processed' # fallback

print(f"Using data directory: {DATA_DIR}")

X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"))
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))
X_val = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"))
y_val = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))
X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))

y_train_binary = y_train["Label_Binary"].values.ravel()
y_train_multi = y_train["Label_Multiclass"].values.ravel()
y_val_binary = y_val["Label_Binary"].values.ravel()
y_val_multi = y_val["Label_Multiclass"].values.ravel()
y_test_binary = y_test["Label_Binary"].values.ravel()
y_test_multi = y_test["Label_Multiclass"].values.ravel()

print("Data loaded successfully!")
print(f"X_train shape: {X_train.shape}")

## 2. Hyperparameter Tuning (Binary Classification)
We use `LinearSVC` as it scales better to large datasets. We'll tune the `C` parameter (regularization strength) using cross-validation on the training set.

In [ ]:
print("Starting Hyperparameter Tuning for Binary Model...")
param_grid = {'C': [0.01, 0.1, 1, 10]}

svm_grid = GridSearchCV(LinearSVC(random_state=42, dual=False, max_iter=2000), 
                        param_grid, 
                        cv=3, 
                        scoring='f1', 
                        n_jobs=-1, 
                        verbose=2)

svm_grid.fit(X_train, y_train_binary)

print(f"Best Parameters: {svm_grid.best_params_}")
print(f"Best CV F1-Score: {svm_grid.best_score_:.4f}")

best_binary_model = svm_grid.best_estimator_

## 3. Evaluation Setup

In [ ]:
def evaluate_model(model, X, y, title_prefix="", is_multiclass=False):
    preds = model.predict(X)
    avg_method = 'weighted' if is_multiclass else 'binary'
    
    print(f"--- {title_prefix} Classification Report ---")
    print(classification_report(y, preds, zero_division=0))
    
    cm = confusion_matrix(y, preds)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{title_prefix} Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    return preds

## 4. Evaluate Binary Model

In [ ]:
_ = evaluate_model(best_binary_model, X_val, y_val_binary, title_prefix="Binary Validation")
_ = evaluate_model(best_binary_model, X_test, y_test_binary, title_prefix="Binary Test")

## 5. Train & Evaluate Multiclass Model
For the multiclass model, we will use the best `C` parameter found during the binary search, or you can re-run `GridSearchCV`.

In [ ]:
best_C = svm_grid.best_params_['C']
print(f"Training Multiclass model with C={best_C}...")

best_multi_model = LinearSVC(C=best_C, random_state=42, dual=False, max_iter=2000)
best_multi_model.fit(X_train, y_train_multi)

_ = evaluate_model(best_multi_model, X_val, y_val_multi, title_prefix="Multiclass Validation", is_multiclass=True)
_ = evaluate_model(best_multi_model, X_test, y_test_multi, title_prefix="Multiclass Test", is_multiclass=True)

## 6. Save Models
Export the best models to the `saved_models` directory.

In [ ]:
if os.path.exists('../../../saved_models'):
    SAVE_DIR = '../../../saved_models'
elif os.path.exists('saved_models'):
    SAVE_DIR = 'saved_models'
else:
    SAVE_DIR = '../saved_models'

os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(best_binary_model, os.path.join(SAVE_DIR, "svm_model_binary.pkl"))
joblib.dump(best_multi_model, os.path.join(SAVE_DIR, "svm_model_multiclass.pkl"))

print("Models saved successfully!")